In [ ]:
import numpy as np
import pandas as pd
import pod_models
import matplotlib.pyplot as plt
from scipy.stats import lognorm
from functools import partial

rng = np.random.default_rng()

plt.rcParams.update({
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 600,
    'figure.constrained_layout.use': True,
})

color_Campbell = "#b6321c" # BrickRed
color_DNVGL = "#f58137" # Orange
color_CV = "#000000"
color_prior = "#0071bc" # RoyalBlue

color_0_5 = "#C0C0C0" # light gray
color_1_5 = "#808080" # medium gray
color_3_0 = "#404040" # dark gray
color_17_0 = "#000000" # black

color_hit  = "#006d02" # green
color_miss = "#a60014" # red
color_fp = "#0043a0" # blue

text_width = 5.3 # Document text width in inches

In [ ]:
# As determined in general_pod.ipynb.
def PoD_base(a):
    return pod_models.PoD(np.log(a[..., None]), -1.890, np.array([0.858]))

# As determined with the parametric approach in resolution_effect.ipynb.
def PoD_surface(a, r):
    return pod_models.PoD(np.stack([np.log(a), np.log(r) * np.ones_like(a)], axis=-1), -3.446, np.array([0.881, 1.058]))

PoD_surface_0_5 = partial(PoD_surface, r=0.5)
PoD_surface_1_5 = partial(PoD_surface, r=1.5)
PoD_surface_3_0 = partial(PoD_surface, r=3.0)
PoD_surface_17_0 = partial(PoD_surface, r=17.0)

means = [37.15, 117.51, 371.72]
coefficients_of_variation = [0.25, 0.5, 1, 2]

In [ ]:
def perform_and_process_MC_crack_length(prior_mean, prior_coefficient_of_variation, num_samples=1000000, batch_size=1000000):
    inspection_methods = dict(
        CV=PoD_base,
        DNVGL=pod_models.DNVGL,
        Campbell=pod_models.Campbell,
    )
    results = pod_models.MC_simulation(inspection_methods, prior_mean, prior_coefficient_of_variation, num_samples=num_samples, batch_size=batch_size, rng=rng)

    cracks = results["generated_cracks"]
    results_CV = results["CV"]["missed_cracks"]
    results_DNVGL = results["DNVGL"]["missed_cracks"]
    results_Campbell = results["Campbell"]["missed_cracks"]

    s_prior, _, scale_prior = lognorm.fit(cracks, floc=0)
    s_CV, _, scale_CV = lognorm.fit(results_CV, floc=0)
    s_DNVGL, _, scale_DNVGL = lognorm.fit(results_DNVGL, floc=0)
    s_Campbell, _, scale_Campbell = lognorm.fit(results_Campbell, floc=0)

    figsize = (0.7 * text_width, 0.7 * 0.5 * text_width)
    # Histograms
    q_low, q_high = 0.01, 0.99
    x_min = max(np.quantile(cracks, q_low), 1e-2)
    x_max = np.quantile(cracks, q_high)
    bins = np.geomspace(x_min, x_max, 51)
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    for data, color, label in [
        (cracks, color_prior, "Prior"),
        (results_CV, color_CV, "CV (ours)"),
        (results_DNVGL, color_DNVGL, "DNVGL"),
        (results_Campbell, color_Campbell, "Campbell"),
    ]:
        ax.hist(data, bins=bins, alpha=0.5, color=color, label=label)
        counts, edges = np.histogram(data, bins=bins)
        ax.hlines(counts, edges[:-1], edges[1:], colors=color, linewidth=2)


    ax.set_title(f"$\\mu = {prior_mean:.2f}$ mm, $\\operatorname{{CoV}} = {prior_coefficient_of_variation}$")
    ax.set_xscale("log")
    ax.set_xlim(x_min, x_max)
    ax.ticklabel_format(axis="y", scilimits=(-1, 1))
    ax.set_xlabel("Crack Length (mm)")
    ax.set_ylabel("Count")
    ax.grid(alpha=0.3)
    ax.legend()

    fig.savefig(f"Figures/update_crack_length_histogram_{prior_mean}_{prior_coefficient_of_variation}.png")

    # Cumulative Distribution Functions
    x_min, x_max = np.min(cracks), np.max(cracks)
    xs = np.geomspace(x_min, x_max, 500)
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.plot(
        xs, lognorm.cdf(xs, s=s_prior, loc=0, scale=scale_prior), color=color_prior, linestyle='-',
        label="Prior"
    )
    ax.plot(
        xs, lognorm.cdf(xs, s=s_CV, loc=0, scale=scale_CV), color=color_CV, linestyle='--',
        label="CV (ours)"
    )
    ax.plot(
        xs, lognorm.cdf(xs, s=s_DNVGL, loc=0, scale=scale_DNVGL), color=color_DNVGL, linestyle='--',
        label="DNVGL"
    )
    ax.plot(
        xs, lognorm.cdf(xs, s=s_Campbell, loc=0, scale=scale_Campbell), color=color_Campbell, linestyle='--',
        label="Campbell"
    )

    ax.set_title(f"$\\mu = {prior_mean:.2f}$ mm, $\\operatorname{{CoV}} = {prior_coefficient_of_variation}$")
    ax.set_xlabel('Crack Length (mm)')
    ax.set_ylabel('Cumulative Probability')
    ax.set_xscale('log')
    ax.set_xlim([x_min, x_max])
    ax.grid(alpha=0.35)
    ax.legend(loc='lower right', framealpha=0.95)

    fig.savefig(f"Figures/update_crack_length_cdf_{prior_mean}_{prior_coefficient_of_variation}.png")



    total_crack_length = np.sum(cracks)
    Cs = dict(
        CV=pod_models.metric_C(np.sum(results_CV), total_crack_length),
        DNVGL=pod_models.metric_C(np.sum(results_DNVGL), total_crack_length),
        Campbell=pod_models.metric_C(np.sum(results_Campbell), total_crack_length),
    )
    KLs = dict(
        CV=pod_models.metric_KL(s_prior, scale_prior, s_CV, scale_CV),
        DNVGL=pod_models.metric_KL(s_prior, scale_prior, s_DNVGL, scale_DNVGL),
        Campbell=pod_models.metric_KL(s_prior, scale_prior, s_Campbell, scale_Campbell),
    )

    return Cs, KLs


## Fig. 9, and Figs. A.18-A.29

In [ ]:
Cs = dict(
    CV=np.zeros((len(means), len(coefficients_of_variation))),
    DNVGL=np.zeros((len(means), len(coefficients_of_variation))),
    Campbell=np.zeros((len(means), len(coefficients_of_variation))),
)
KLs = dict(
    CV=np.zeros((len(means), len(coefficients_of_variation))),
    DNVGL=np.zeros((len(means), len(coefficients_of_variation))),
    Campbell=np.zeros((len(means), len(coefficients_of_variation))),
)
for i, mean in enumerate(means):
    for j, coefficient_of_variation in enumerate(coefficients_of_variation):
        print(f"mean = {mean}, coefficient of variation = {coefficient_of_variation}")
        C, KL = perform_and_process_MC_crack_length(mean, coefficient_of_variation)
        for name in C:
            Cs[name][i, j] = C[name]
            KLs[name][i, j] = KL[name]

# Table 1

In [ ]:
with open("Tables/update_crack_length.tex", "w") as f:
    for i, mu in enumerate(means):

        for j, cov in enumerate(coefficients_of_variation):

            mu_col = rf"\multirow{{4}}{{*}}{{{mu}}}" if j == 0 else ""

            f.write(
                f"{mu_col} & {cov}"
                f" & {Cs['CV'][i, j]:.3f}"
                f" & {Cs['Campbell'][i, j]:.3f}"
                f" & {Cs['DNVGL'][i, j]:.3f}"
                f" & {KLs['CV'][i, j]:.3f}"
                f" & {KLs['Campbell'][i, j]:.3f}"
                f" & {KLs['DNVGL'][i, j]:.3f}"
                r" \\"
                "\n"
            )

        if i < len(means) - 1:
            f.write("\\hline\n")

# Figs. 7, 8

In [ ]:
rows_C = []
rows_KL = []
for fig in range(3):
    for j, coefficient_of_variation in enumerate(coefficients_of_variation):
        rows_C.append({
            "fig": fig,
            "CoV": coefficient_of_variation,
            "CV": Cs['CV'][fig, j],
            "Campbell": Cs['Campbell'][fig, j],
            "DNVGL": Cs['DNVGL'][fig, j],
        })
        rows_KL.append({
            "fig": fig,
            "CoV": coefficient_of_variation,
            "CV": KLs['CV'][fig, j],
            "Campbell": KLs['Campbell'][fig, j],
            "DNVGL": KLs['DNVGL'][fig, j],
        })


pd.DataFrame(rows_C).to_csv("Figures/update_crack_length_C.dat", sep=" ", index=False)
pd.DataFrame(rows_KL).to_csv("Figures/update_crack_length_KL.dat", sep=" ", index=False)

In [ ]:
def perform_and_process_MC_crack_length_resolution(prior_mean, prior_coefficient_of_variation, num_samples=1000000, batch_size=1000000):
    inspection_methods = dict(
        CV_17_0=PoD_surface_17_0,
        CV_3_0=PoD_surface_3_0,
        CV_1_5=PoD_surface_1_5,
        CV_0_5=PoD_surface_0_5,
        DNVGL=pod_models.DNVGL,
        Campbell=pod_models.Campbell,
    )
    results = pod_models.MC_simulation(inspection_methods, prior_mean, prior_coefficient_of_variation, num_samples=num_samples, batch_size=batch_size, rng=rng)

    cracks = results["generated_cracks"]
    results_17_0 = results["CV_17_0"]["missed_cracks"]
    results_3_0 = results["CV_3_0"]["missed_cracks"]
    results_1_5 = results["CV_1_5"]["missed_cracks"]
    results_0_5 = results["CV_0_5"]["missed_cracks"]
    results_DNVGL = results["DNVGL"]["missed_cracks"]
    results_Campbell = results["Campbell"]["missed_cracks"]

    s_prior, _, scale_prior = lognorm.fit(cracks, floc=0)
    s_17_0, _, scale_17_0 = lognorm.fit(results_17_0, floc=0)
    s_3_0, _, scale_3_0 = lognorm.fit(results_3_0, floc=0)
    s_1_5, _, scale_1_5 = lognorm.fit(results_1_5, floc=0)
    s_0_5, _, scale_0_5 = lognorm.fit(results_0_5, floc=0)
    s_DNVGL, _, scale_DNVGL = lognorm.fit(results_DNVGL, floc=0)
    s_Campbell, _, scale_Campbell = lognorm.fit(results_Campbell, floc=0)

    total_crack_length = np.sum(cracks)
    Cs = dict(
        CV_17_0=pod_models.metric_C(np.sum(results_17_0), total_crack_length),
        CV_3_0=pod_models.metric_C(np.sum(results_3_0), total_crack_length),
        CV_1_5=pod_models.metric_C(np.sum(results_1_5), total_crack_length),
        CV_0_5=pod_models.metric_C(np.sum(results_0_5), total_crack_length),
        DNVGL=pod_models.metric_C(np.sum(results_DNVGL), total_crack_length),
        Campbell=pod_models.metric_C(np.sum(results_Campbell), total_crack_length),
    )
    KLs = dict(
        CV_17_0=pod_models.metric_KL(s_prior, scale_prior, s_17_0, scale_17_0),
        CV_3_0=pod_models.metric_KL(s_prior, scale_prior, s_3_0, scale_3_0),
        CV_1_5=pod_models.metric_KL(s_prior, scale_prior, s_1_5, scale_1_5),
        CV_0_5=pod_models.metric_KL(s_prior, scale_prior, s_0_5, scale_0_5),
        DNVGL=pod_models.metric_KL(s_prior, scale_prior, s_DNVGL, scale_DNVGL),
        Campbell=pod_models.metric_KL(s_prior, scale_prior, s_Campbell, scale_Campbell),
    )

    return Cs, KLs

In [ ]:
Cs = dict(
    CV_17_0=np.zeros((len(means), len(coefficients_of_variation))),
    CV_3_0=np.zeros((len(means), len(coefficients_of_variation))),
    CV_1_5=np.zeros((len(means), len(coefficients_of_variation))),
    CV_0_5=np.zeros((len(means), len(coefficients_of_variation))),
    DNVGL=np.zeros((len(means), len(coefficients_of_variation))),
    Campbell=np.zeros((len(means), len(coefficients_of_variation))),
)
KLs = dict(
    CV_17_0=np.zeros((len(means), len(coefficients_of_variation))),
    CV_3_0=np.zeros((len(means), len(coefficients_of_variation))),
    CV_1_5=np.zeros((len(means), len(coefficients_of_variation))),
    CV_0_5=np.zeros((len(means), len(coefficients_of_variation))),
    DNVGL=np.zeros((len(means), len(coefficients_of_variation))),
    Campbell=np.zeros((len(means), len(coefficients_of_variation))),
)
for i, mean in enumerate(means):
    for j, coefficient_of_variation in enumerate(coefficients_of_variation):
        print(f"mean = {mean}, coefficient of variation = {coefficient_of_variation}")
        C, KL = perform_and_process_MC_crack_length_resolution(mean, coefficient_of_variation)
        for name in C:
            Cs[name][i, j] = C[name]
            KLs[name][i, j] = KL[name]

## Tabs. 2, 3

In [ ]:
with open("Tables/update_resolution_C.tex", "w") as f:
    for i, mu in enumerate(means):

        for j, cov in enumerate(coefficients_of_variation):

            mu_col = rf"\multirow{{4}}{{*}}{{{mu}}}" if j == 0 else ""

            f.write(
                f"{mu_col} & {cov}"
                f" & {Cs['CV_17_0'][i, j]:.3f}"
                f" & {Cs['CV_3_0'][i, j]:.3f}"
                f" & {Cs['CV_1_5'][i, j]:.3f}"
                f" & {Cs['CV_0_5'][i, j]:.3f}"
                f" & {Cs['Campbell'][i, j]:.3f}"
                f" & {Cs['DNVGL'][i, j]:.3f}"
                r" \\"
                "\n"
            )

        if i < len(means) - 1:
            f.write("\\hline\n")

with open("Tables/update_resolution_KL.tex", "w") as f:
    for i, mu in enumerate(means):

        for j, cov in enumerate(coefficients_of_variation):

            mu_col = rf"\multirow{{4}}{{*}}{{{mu}}}" if j == 0 else ""

            f.write(
                f"{mu_col} & {cov}"
                f" & {KLs['CV_17_0'][i, j]:.3f}"
                f" & {KLs['CV_3_0'][i, j]:.3f}"
                f" & {KLs['CV_1_5'][i, j]:.3f}"
                f" & {KLs['CV_0_5'][i, j]:.3f}"
                f" & {KLs['Campbell'][i, j]:.3f}"
                f" & {KLs['DNVGL'][i, j]:.3f}"
                r" \\"
                "\n"
            )

        if i < len(means) - 1:
            f.write("\\hline\n")

## Figs. 16, 17

In [ ]:
rows_C = []
rows_KL = []
for fig in range(3):
    for j, coefficient_of_variation in enumerate(coefficients_of_variation):
        rows_C.append({
            "fig": fig,
            "CoV": coefficient_of_variation,
            "CV_17_0": Cs['CV_17_0'][fig, j],
            "CV_3_0": Cs['CV_3_0'][fig, j],
            "CV_1_5": Cs['CV_1_5'][fig, j],
            "CV_0_5": Cs['CV_0_5'][fig, j],
            "Campbell": Cs['Campbell'][fig, j],
            "DNVGL": Cs['DNVGL'][fig, j],
        })
        rows_KL.append({
            "fig": fig,
            "CoV": coefficient_of_variation,
            "CV_17_0": KLs['CV_17_0'][fig, j],
            "CV_3_0": KLs['CV_3_0'][fig, j],
            "CV_1_5": KLs['CV_1_5'][fig, j],
            "CV_0_5": KLs['CV_0_5'][fig, j],
            "Campbell": KLs['Campbell'][fig, j],
            "DNVGL": KLs['DNVGL'][fig, j],
        })


pd.DataFrame(rows_C).to_csv("Figures/update_resolution_C.dat", sep=" ", index=False)
pd.DataFrame(rows_KL).to_csv("Figures/update_resolution_KL.dat", sep=" ", index=False)